In [1]:
import pandas as pd
import random

# Gerçek kurumlar yerine güvenli, kurgusal ama gerçekçi ana domainler (Noise yaratmak için)
legit_domains = [
    "trusted-bank.com", 
    "state-health.gov", 
    "city-university.edu", 
    "express-delivery.com", 
    "global-shop.net"
]

# Dolandırıcıların sık kullandığı tetikleyici (suspicious) kelimeler
suspicious_keywords = ["login", "onay", "update", "hesap", "odeme", "sifre"]

print("Kütüphaneler yüklendi ve simülasyon havuzu hazır!")

Kütüphaneler yüklendi ve simülasyon havuzu hazır!


In [2]:
def generate_url_dataset(n_samples=20):
    data = []
    
    for _ in range(n_samples):
        # 0: Legitimate (Normal), 1: Fraud (Dolandırıcı)
        label = random.choice([0, 1])
        
        # Temel değişkenleri başlatalım
        domain = random.choice(legit_domains)
        suspicious_keyword_present = 0
        hyphen_count = 0
        
        if label == 0: 
            # NORMAL SİTELER (Legitimate)
            fraud_type = "Legitimate"
            # Normal sitelerin %95'i https kullanır, %5'i http (Gerçekçi gürültü)
            protocol = "https" if random.random() < 0.95 else "http"
            has_https = 1 if protocol == "https" else 0
            
            url = f"{protocol}://www.{domain}"
            
        else: 
            # DOLANDIRICI SİTELER (Fraud)
            fraud_type = "Phishing"
            # Dolandırıcıların %60'ı http, %40'ı https kullanır (Veri sızıntısını önler)
            protocol = "https" if random.random() < 0.40 else "http"
            has_https = 1 if protocol == "https" else 0
            
            base_domain = domain.split('.')[0] # Örn: trusted-bank
            
            # Rastgele şüpheli kelime veya tire ekleme
            if random.random() < 0.7:
                keyword = random.choice(suspicious_keywords)
                suspicious_keyword_present = 1
                hyphen_count = 1
                url = f"{protocol}://www.{base_domain}-{keyword}.com"
            else:
                # Şüpheli kelime havuzunda olmayan başka bir kalıp (Model sadece kelime ezberlemesin diye)
                hyphen_count = 2
                url = f"{protocol}://www.{base_domain}-secure-update.net"
                
        # String'in toplam uzunluğu
        url_length = len(url)
        
        # Sözlük yapısında satırı oluştur
        data.append({
            "url": url,
            "label": label,
            "fraud_type": fraud_type,
            "has_https": has_https,
            "suspicious_keyword": suspicious_keyword_present,
            "hyphen_count": hyphen_count,
            "url_length": url_length
        })
        
    return pd.DataFrame(data)

# Fonksiyonu test edip ilk 10 satırı tablo olarak görelim
df_urls = generate_url_dataset(n_samples=10)
display(df_urls)

,url,label,fraud_type,has_https,suspicious_keyword,hyphen_count,url_length
0,https://www.trusted-bank-secure-update.net,1,Phishing,1,0,2,42
1,https://www.city-university.edu,0,Legitimate,1,0,0,31
2,https://www.express-delivery.com,0,Legitimate,1,0,0,32
3,http://www.city-university-login.com,1,Phishing,0,1,1,36
4,https://www.state-health.gov,0,Legitimate,1,0,0,28
5,https://www.global-shop.net,0,Legitimate,1,0,0,27
6,https://www.trusted-bank-login.com,1,Phishing,1,1,1,34
7,https://www.state-health.gov,0,Legitimate,1,0,0,28
8,http://www.global-shop-sifre.com,1,Phishing,0,1,1,32
9,https://www.express-delivery-hesap.com,1,Phishing,1,1,1,38


In [3]:
# 1000 satırlık ana veri setini üret
df_url_raw = generate_url_dataset(n_samples=1000)

# Veriyi oluşturduğumuz klasör yapısına uygun olarak 'data/raw' içine CSV olarak kaydet
# Not: Notebook'umuz 'notebooks' klasörü içinde olduğu için bir üst klasöre (..) çıkıp data'ya giriyoruz.
output_path = "../data/raw/url_dataset.csv"
df_url_raw.to_csv(output_path, index=False)

print(f"Veri seti başarıyla kaydedildi: {output_path}")
print(f"Toplam Satır ve Sütun Sayısı: {df_url_raw.shape}")

# Dağılıma hızlıca bir göz atalım
print("\nSınıf Dağılımı (0: Normal, 1: Fraud):")
print(df_url_raw['label'].value_counts())

Veri seti başarıyla kaydedildi: ../data/raw/url_dataset.csv
Toplam Satır ve Sütun Sayısı: (1000, 7)

Sınıf Dağılımı (0: Normal, 1: Fraud):
label
0    517
1    483
Name: count, dtype: int64
